# Phase 5 adapter integration
Run this notebook twice in separate fresh GPU runtimes: family `ttm`, then `moirai1`. Python 3.11/3.12. FP32, ETTh1 train/validation, batch 2, one step. No test evaluation or pilot. Prior probe success does not imply adapter success. Execute cells in order; retain failed JSON. Drive authentication is a user action.

In [ ]:
import hashlib
import json
import os
import subprocess
import sys
import urllib.request
import zipfile
from pathlib import Path

assert (3, 11) <= sys.version_info[:2] <= (3, 12), "Select Python 3.11/3.12 runtime"
subprocess.run(["nvidia-smi"], check=True)
print("Kernel:", sys.executable, sys.version)
FAMILY = input("Model family (ttm or moirai1): ").strip()
assert FAMILY in ("ttm", "moirai1")
COMMIT = input("Full published phase-5 adapter commit SHA: ").strip()
assert len(COMMIT) == 40 and all(c in "0123456789abcdef" for c in COMMIT)

In [ ]:
ROOT = Path("/content") / ("tsfm-adapter-" + COMMIT)
URL = "https://github.com/Han-Youseung/tsfm-zero-few-shot-crossover.git"
if not ROOT.exists():
    subprocess.run(["git", "clone", URL, str(ROOT)], check=True)
assert (
    subprocess.check_output(
        ["git", "-C", str(ROOT), "remote", "get-url", "origin"], text=True
    ).strip()
    == URL
)
assert not subprocess.check_output(
    ["git", "-C", str(ROOT), "status", "--porcelain"], text=True
).strip()
subprocess.run(["git", "-C", str(ROOT), "checkout", "--detach", COMMIT], check=True)
os.chdir(ROOT)
assert subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip() == COMMIT
print("Immutable execution commit:", COMMIT)
# Do not pull or edit source during this execution.

In [ ]:
USE_DRIVE = True  # User executes mount/authentication; set False for download-only.
if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    OUT = Path("/content/drive/MyDrive/tsfm-adapter-integration") / COMMIT / FAMILY
else:
    OUT = ROOT / "results/raw/adapters" / COMMIT / FAMILY
OUT.mkdir(parents=True, exist_ok=True)
print("Destination:", OUT, "\nTemporary runtime storage is not persistent.")

In [ ]:
ENV = Path("/content") / ("venv-adapter-" + FAMILY)
PY = ENV / "bin/python"
if not PY.exists():
    subprocess.run(["apt-get", "update"], check=True)
    subprocess.run(
        [
            "apt-get",
            "install",
            "-y",
            f"python{sys.version_info.major}.{sys.version_info.minor}-venv",
        ],
        check=True,
    )
    subprocess.run([sys.executable, "-m", "venv", str(ENV)], check=True)
subprocess.run(
    [str(PY), "-m", "pip", "install", "-r", f"requirements/{FAMILY}-gpu.txt"], check=True
)
subprocess.run([str(PY), "-m", "pip", "install", "-e", ".[dev]"], check=True)
subprocess.run([str(PY), "-m", "pip", "check"], check=True)
# Installation and every model process use exactly PY; kernel imports no model package.
code = (
    "import sys, torch; print('Probe:',sys.executable,sys.version,"
    "torch.__version__,torch.version.cuda); assert torch.cuda.is_available();"
    "print(torch.cuda.get_device_name())"
)
subprocess.run([str(PY), "-c", code], check=True)

In [ ]:
DATA = ROOT / "data/official_raw/ETTh1.csv"
DATA.parent.mkdir(parents=True, exist_ok=True)
if not DATA.exists():
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/1d16c8f4f943005d613b5bc962e9eeb06058cf07/ETT-small/ETTh1.csv",
        DATA,
    )
assert (
    hashlib.sha256(DATA.read_bytes()).hexdigest()
    == "f18de3ad269cef59bb07b5438d79bb3042d3be49bdeecf01c1cd6d29695ee066"
)
print("Official ETTh1 fingerprint verified")

In [ ]:
for horizon in (96, 192, 336, 720):
    dest = OUT / f"{FAMILY}-adapter-h{horizon}.json"
    command = [
        str(PY),
        "scripts/smoke_adapters.py",
        "--config",
        f"configs/adapters/{FAMILY}_smoke.yaml",
        "--data",
        str(DATA),
        "--device",
        "cuda",
        "--horizon",
        str(horizon),
        "--expected-commit",
        COMMIT,
        "--output",
        str(dest),
    ]
    completed = subprocess.run(command, check=False)
    print(
        horizon,
        "exit:",
        completed.returncode,
        "status:",
        json.loads(dest.read_text())["status"] if dest.exists() else "missing",
    )
# Existing passed conditions resume. Failed attempts are never overwritten; preserve them
# and choose a new destination after diagnosing a failure.

In [ ]:
from google.colab import files

archive = Path("/content") / f"{FAMILY}-adapter-results.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(OUT.glob("*.json")):
        assert path.stat().st_size < 2000000
        bundle.write(path, arcname=path.name)
files.download(str(archive))
print("Download includes failed attempts. Return this ZIP for source/identity/check validation.")